## Step 8: Regime Segmentation Comparison

Split into sub-periods and test whether relationships hold across different market regimes.

In [ ]:
print("\n" + "=" * 70)
print("LAG CORRELATION ANALYSIS")
print("=" * 70)

MAX_LAG = 5

# Create a time series indexed by date for lagged analysis
sent_ts = sentiment_dispersion.set_index("article_date")[["sentiment_var"]].sort_index()
ret_ts = return_dispersion.set_index("price_date")[["return_var"]].sort_index()
corr_ts = rolling_correlations.set_index("price_date")[["corr_mean"]].sort_index()

lag_results = []

for lag in range(0, MAX_LAG + 1):
    # Shift sentiment forward by `lag` days to test lagged relationship
    sent_shifted = sent_ts.copy()
    sent_shifted.index = sent_shifted.index + pd.Timedelta(days=lag)
    
    # Merge with returns at this lag
    lag_merged_ret = sent_shifted.merge(ret_ts, left_index=True, right_index=True, how="inner")
    lag_merged_corr = sent_shifted.merge(corr_ts, left_index=True, right_index=True, how="inner")
    
    # H1: sentiment_var vs return_var at this lag
    if len(lag_merged_ret) > 2:
        rho_ret, p_ret = spearmanr(lag_merged_ret["sentiment_var"], lag_merged_ret["return_var"])
    else:
        rho_ret, p_ret = np.nan, np.nan
    
    # H2: sentiment_var vs corr_mean at this lag
    if len(lag_merged_corr) > 2:
        rho_corr, p_corr = spearmanr(lag_merged_corr["sentiment_var"], lag_merged_corr["corr_mean"])
    else:
        rho_corr, p_corr = np.nan, np.nan
    
    lag_results.append({
        "lag_days": lag,
        "n_ret": len(lag_merged_ret),
        "rho_return_disp": rho_ret,
        "p_return_disp": p_ret,
        "n_corr": len(lag_merged_corr),
        "rho_corr_breakdown": rho_corr,
        "p_corr_breakdown": p_corr,
    })
    print(f"  Lag {lag}d: return_disp ρ={rho_ret:+.4f} (p={p_ret:.4f}, n={len(lag_merged_ret)}) | "
          f"corr_breakdown ρ={rho_corr:+.4f} (p={p_corr:.4f}, n={len(lag_merged_corr)})")

lag_df = pd.DataFrame(lag_results)

# Identify optimal lags
best_lag_ret = lag_df.loc[lag_df["rho_return_disp"].abs().idxmax()]
best_lag_corr = lag_df.loc[lag_df["rho_corr_breakdown"].abs().idxmax()]

print(f"\nBest lag for H1 (return dispersion): Lag {int(best_lag_ret['lag_days'])}d, ρ={best_lag_ret['rho_return_disp']:.4f}")
print(f"Best lag for H2 (corr breakdown):    Lag {int(best_lag_corr['lag_days'])}d, ρ={best_lag_corr['rho_corr_breakdown']:.4f}")

# Lag correlation bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_ret = ['steelblue' if p < 0.05 else 'lightgray' for p in lag_df["p_return_disp"]]
axes[0].bar(lag_df["lag_days"], lag_df["rho_return_disp"], color=colors_ret, edgecolor='gray')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel("Lag (days)", fontsize=11)
axes[0].set_ylabel("Spearman ρ", fontsize=11)
axes[0].set_title("H1: Sentiment Disp → Return Disp\n(blue = p < 0.05)", fontsize=12, fontweight='bold')
axes[0].set_xticks(range(MAX_LAG + 1))
axes[0].grid(True, alpha=0.3, axis='y')

colors_corr = ['darkorange' if p < 0.05 else 'lightgray' for p in lag_df["p_corr_breakdown"]]
axes[1].bar(lag_df["lag_days"], lag_df["rho_corr_breakdown"], color=colors_corr, edgecolor='gray')
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel("Lag (days)", fontsize=11)
axes[1].set_ylabel("Spearman ρ", fontsize=11)
axes[1].set_title("H2: Sentiment Disp → Corr Breakdown\n(orange = p < 0.05)", fontsize=12, fontweight='bold')
axes[1].set_xticks(range(MAX_LAG + 1))
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Display full lag table
print("\nFull Lag Correlation Table:")
print(lag_df.to_string(index=False))

## Step 7: Lag Correlation Analysis

Test sentiment dispersion → return dispersion at different lags (0 to 5 days)

In [ ]:
print("\n" + "=" * 70)
print("HYPOTHESIS H3: Quantile Comparison (Top 20% vs Bottom 20%)")
print("=" * 70)

# Split into quantiles based on sentiment_var
h3_data = merged_data[["sentiment_var", "return_var"]].dropna()

if len(h3_data) > 4:
    q80 = h3_data["sentiment_var"].quantile(0.80)
    q20 = h3_data["sentiment_var"].quantile(0.20)
    
    top_20_pct = h3_data[h3_data["sentiment_var"] >= q80]["return_var"]
    bot_20_pct = h3_data[h3_data["sentiment_var"] <= q20]["return_var"]
    
    print(f"\nQuantile thresholds:")
    print(f"  Bottom 20% (≤{q20:.6f}): {len(bot_20_pct)} days")
    print(f"  Top 20% (≥{q80:.6f}): {len(top_20_pct)} days")
    
    print(f"\nReturn Dispersion Statistics:")
    print(f"  Bottom 20% - Mean: {bot_20_pct.mean():.6f}, Median: {bot_20_pct.median():.6f}, Std: {bot_20_pct.std():.6f}")
    print(f"  Top 20%    - Mean: {top_20_pct.mean():.6f}, Median: {top_20_pct.median():.6f}, Std: {top_20_pct.std():.6f}")
    print(f"  Difference (Top - Bottom): {top_20_pct.mean() - bot_20_pct.mean():.6f}")
    
    # Mann-Whitney U test
    if len(top_20_pct) > 0 and len(bot_20_pct) > 0:
        h3_stat, h3_pval = mannwhitneyu(top_20_pct, bot_20_pct, alternative="two-sided")
        print(f"\nMann-Whitney U Test:")
        print(f"  U-statistic = {h3_stat:.4f}")
        print(f"  p-value = {h3_pval:.6f}")
        print(f"  Significant at α=0.05? {h3_pval < 0.05}")
        
        # Effect size: rank-biserial correlation
        n1, n2 = len(top_20_pct), len(bot_20_pct)
        r = 1 - (2*h3_stat)/(n1*n2)
        print(f"  Rank-biserial correlation (effect size): {r:.4f}")
        
        # Conclusion
        if h3_pval < 0.05:
            h3_result = "SUPPORTED"
        else:
            h3_result = "WEAK/UNSTABLE"
        
        print(f"\n  ✓ H3 Result: [{h3_result}]")
        
        # Box plot comparison
        fig, ax = plt.subplots(figsize=(10, 6))
        data_to_plot = [bot_20_pct, top_20_pct]
        bp = ax.boxplot(data_to_plot, labels=["Bottom 20%\nSentiment Disp", "Top 20%\nSentiment Disp"], patch_artist=True)
        for patch, color in zip(bp['boxes'], ['lightblue', 'lightcoral']):
            patch.set_facecolor(color)
        ax.set_ylabel("Return Dispersion (Variance)", fontsize=11)
        ax.set_title(f"H3: Return Dispersion by Sentiment Dispersion Quantiles\n(U={h3_stat:.1f}, p={h3_pval:.4f})", 
                     fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        plt.show()
    else:
        print("Insufficient data for Mann-Whitney test")
        h3_result = "INSUFFICIENT DATA"
else:
    print("Insufficient data for H3 test")
    h3_result = "INSUFFICIENT DATA"

### H3: Quantile Comparison (Top 20% vs Bottom 20% Sentiment Dispersion)
**Hypothesis**: High sentiment dispersion days have significantly higher return dispersion  
**Test**: Mann-Whitney U test (non-parametric)  
**Expected**: Significant difference between groups, p < 0.05

In [ ]:
print("\n" + "=" * 70)
print("HYPOTHESIS H2: Sentiment Dispersion → Correlation Breakdown")
print("=" * 70)

# H2 Test: Spearman correlation between sentiment_var and corr_mean (inverse expected)
h2_data = merged_with_corr[["sentiment_var", "corr_mean"]].dropna()

if len(h2_data) > 2:
    h2_corr, h2_pval = spearmanr(h2_data["sentiment_var"], h2_data["corr_mean"])
    print(f"\nSpearman Correlation Test:")
    print(f"  n = {len(h2_data)}")
    print(f"  ρ = {h2_corr:.4f}")
    print(f"  p-value = {h2_pval:.6f}")
    print(f"  Significant at α=0.05? {h2_pval < 0.05}")
    
    # Effect size interpretation
    if abs(h2_corr) < 0.3:
        effect = "weak"
    elif abs(h2_corr) < 0.7:
        effect = "moderate"
    else:
        effect = "strong"
    print(f"  Effect size: {effect}")
    
    # Direction (H2 expects negative)
    if h2_corr < 0:
        direction = "negative (supports H2)"
    else:
        direction = "positive (contradicts H2)"
    print(f"  Direction: {direction}")
    
    # Conclusion
    if h2_pval < 0.05 and h2_corr < 0:
        h2_result = "SUPPORTED"
    elif h2_pval < 0.05 and h2_corr >= 0:
        h2_result = "REJECTED"
    else:
        h2_result = "WEAK/UNSTABLE"
    
    print(f"\n  ✓ H2 Result: [{h2_result}]")
    
    # Scatter plot
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(h2_data["sentiment_var"], h2_data["corr_mean"], alpha=0.5, s=30, color='orange')
    z = np.polyfit(h2_data["sentiment_var"], h2_data["corr_mean"], 1)
    p = np.poly1d(z)
    ax.plot(h2_data["sentiment_var"], p(h2_data["sentiment_var"]), "r--", linewidth=2, alpha=0.8, label=f"ρ={h2_corr:.3f}, p={h2_pval:.4f}")
    ax.set_xlabel("Sentiment Dispersion (Variance)", fontsize=11)
    ax.set_ylabel("Pairwise Correlation (Mean, rolling 21-day)", fontsize=11)
    ax.set_title("H2: Sentiment Dispersion → Correlation Breakdown", fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Insufficient data for H2 test")
    h2_result = "INSUFFICIENT DATA"

### H2: Sentiment Dispersion → Correlation Breakdown
**Hypothesis**: ↑ sentiment dispersion → ↓ pairwise correlation (more decoupled returns)  
**Test**: Spearman rank correlation (inverse relationship)  
**Expected**: Negative correlation, p < 0.05

In [ ]:
print("\n" + "=" * 70)
print("HYPOTHESIS H1: Sentiment Dispersion → Return Dispersion")
print("=" * 70)

# H1 Test: Spearman correlation between sentiment_var and return_var
h1_data = merged_data[["sentiment_var", "return_var"]].dropna()

if len(h1_data) > 2:
    h1_corr, h1_pval = spearmanr(h1_data["sentiment_var"], h1_data["return_var"])
    print(f"\nSpearman Correlation Test:")
    print(f"  n = {len(h1_data)}")
    print(f"  ρ = {h1_corr:.4f}")
    print(f"  p-value = {h1_pval:.6f}")
    print(f"  Significant at α=0.05? {h1_pval < 0.05}")
    
    # Effect size interpretation
    if abs(h1_corr) < 0.3:
        effect = "weak"
    elif abs(h1_corr) < 0.7:
        effect = "moderate"
    else:
        effect = "strong"
    print(f"  Effect size: {effect}")
    
    # Direction
    if h1_corr > 0:
        direction = "positive (supports H1)"
    else:
        direction = "negative (contradicts H1)"
    print(f"  Direction: {direction}")
    
    # Conclusion
    if h1_pval < 0.05 and h1_corr > 0:
        h1_result = "SUPPORTED"
    elif h1_pval < 0.05 and h1_corr <= 0:
        h1_result = "REJECTED"
    else:
        h1_result = "WEAK/UNSTABLE"
    
    print(f"\n  ✓ H1 Result: [{h1_result}]")
    
    # Scatter plot
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(h1_data["sentiment_var"], h1_data["return_var"], alpha=0.5, s=30)
    z = np.polyfit(h1_data["sentiment_var"], h1_data["return_var"], 1)
    p = np.poly1d(z)
    ax.plot(h1_data["sentiment_var"], p(h1_data["sentiment_var"]), "r--", linewidth=2, alpha=0.8, label=f"ρ={h1_corr:.3f}, p={h1_pval:.4f}")
    ax.set_xlabel("Sentiment Dispersion (Variance)", fontsize=11)
    ax.set_ylabel("Return Dispersion (Variance, next day)", fontsize=11)
    ax.set_title("H1: Sentiment Dispersion → Return Dispersion", fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Insufficient data for H1 test")
    h1_result = "INSUFFICIENT DATA"

## Step 6: Hypothesis Testing

### H1: Sentiment Dispersion → Return Dispersion
**Hypothesis**: ↑ sentiment dispersion (article_date) → ↑ return dispersion (next trading day)  
**Test**: Spearman rank correlation  
**Expected**: Positive correlation, p < 0.05

In [ ]:
# Align sentiment and return dispersion
# Sentiment on article_date → paired with returns on next trading day (article_date + 1)

# Rename dates for merging
sentiment_for_merge = sentiment_dispersion.copy()
sentiment_for_merge.columns = ["sentiment_date"] + sentiment_for_merge.columns[1:].tolist()
sentiment_for_merge["next_date"] = sentiment_for_merge["sentiment_date"] + pd.Timedelta(days=1)

# Right join: for each return date, find nearest sentiment date
merged_data = return_dispersion.merge(
    sentiment_for_merge,
    left_on="price_date",
    right_on="next_date",
    how="left"
).sort_values("price_date")

# Keep only rows where we have both sentiment and return data
merged_data = merged_data[
    (merged_data["sentiment_var"].notna()) & 
    (merged_data["return_var"].notna())
].reset_index(drop=True)

print("\n" + "=" * 70)
print("MERGED DATASET (Sentiment → Next-Day Returns)")
print("=" * 70)
print(f"Merged rows: {len(merged_data):,}")
print(f"Date range: {merged_data['price_date'].min().date()} to {merged_data['price_date'].max().date()}")
print(f"\nFirst few rows:")
print(merged_data[["sentiment_date", "price_date", "sentiment_var", "return_var", "corr_mean"]].head(10))

# Also merge with rolling correlations
merged_with_corr = merged_data.merge(
    rolling_correlations,
    on="price_date",
    how="left"
)

merged_with_corr = merged_with_corr[merged_with_corr["corr_mean"].notna()].reset_index(drop=True)

print(f"\nMerged with rolling correlations: {len(merged_with_corr):,} rows")
print("\n✓ Step 5 complete: Datasets aligned")

## Step 5: Align Datasets for Hypothesis Testing

Merge sentiment (by article_date) with returns (by price_date) with appropriate lags.

In [ ]:
# For correlations, use a rolling window approach (e.g., 21-day rolling correlation)
# This gives us daily correlation snapshots based on recent returns

ROLLING_WINDOW = 21  # 21-day window for rolling correlation

# Create pivot table: rows=dates, columns=tickers, values=returns
returns_pivot = daily_returns.pivot_table(
    index="price_date", 
    columns="ticker", 
    values="next_day_ret"
)

print("\n" + "=" * 70)
print("PAIRWISE CORRELATION ANALYSIS")
print("=" * 70)
print(f"\nReturns pivot shape: {returns_pivot.shape}")
print(f"Missing values per ticker:")
print(returns_pivot.isna().sum())

# Compute rolling correlations using a 21-day window
def compute_rolling_correlation(returns_df, window=21):
    """Compute correlation metrics from rolling correlation matrices"""
    corr_means = []
    corr_medians = []
    corr_stds = []
    dates = []
    
    for i in range(window - 1, len(returns_df)):
        window_data = returns_df.iloc[i - window + 1:i + 1]
        corr_matrix = window_data.corr()
        
        # Extract upper triangle of correlation matrix (exclude diagonal)
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
        corr_values = corr_matrix.values[mask]
        
        # Remove NaNs
        corr_values = corr_values[~np.isnan(corr_values)]
        
        if len(corr_values) > 0:
            corr_means.append(corr_values.mean())
            corr_medians.append(np.median(corr_values))
            corr_stds.append(corr_values.std())
            dates.append(returns_df.index[i])
    
    return pd.DataFrame({
        "price_date": dates,
        "corr_mean": corr_means,
        "corr_median": corr_medians,
        "corr_std": corr_stds,
    })

rolling_correlations = compute_rolling_correlation(returns_pivot, window=ROLLING_WINDOW)

print(f"\nRolling correlation days: {len(rolling_correlations):,}")
print(f"Correlation statistics:")
print(rolling_correlations[["corr_mean", "corr_median", "corr_std"]].describe())

# Visualize rolling correlations
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(rolling_correlations["price_date"], rolling_correlations["corr_mean"], 
             color='darkblue', linewidth=1.5, label='Mean correlation', alpha=0.8)
axes[0].plot(rolling_correlations["price_date"], rolling_correlations["corr_median"], 
             color='purple', linewidth=1, linestyle='--', label='Median correlation', alpha=0.7)
axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0].set_ylabel("Correlation", fontsize=11)
axes[0].set_title(f"Rolling {ROLLING_WINDOW}-day Pairwise Returns Correlation (MAG7)", fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(rolling_correlations["price_date"], rolling_correlations["corr_std"], 
             color='navy', linewidth=1)
axes[1].set_ylabel("Correlation Std Dev", fontsize=11)
axes[1].set_xlabel("Price Date", fontsize=11)
axes[1].set_title("Rolling Correlation Dispersion", fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Step 4 complete: Rolling pairwise correlations computed")

## Step 4: Compute Pairwise Correlation per Day

In [ ]:
# Reload full data for return calculation (need all MAG7, no sentiment filter)
df_ret = pd.read_parquet("data/processed/gdelt_ohlcv_join.parquet")

# Filter to MAG7 + date window (no sentiment filter needed for prices)
df_ret = df_ret[
    (df_ret["ticker"].isin(MAG7)) &
    (df_ret["article_date"] >= WINDOW_START) &
    (df_ret["article_date"] <= WINDOW_END)
].copy()

print("\n" + "=" * 70)
print("RETURN DISPERSION CALCULATION")
print("=" * 70)
print(f"\nRows for return calculation: {len(df_ret):,}")

# Calculate one-day returns: (next_close - next_open) / next_open
df_ret["next_day_ret"] = (df_ret["next_close"] - df_ret["next_open"]) / df_ret["next_open"]

# Get one row per (price_date, ticker) — all articles on same date have same return
daily_returns = (
    df_ret[["price_date", "ticker", "next_day_ret"]]
    .drop_duplicates(subset=["price_date", "ticker"], keep="first")
    .reset_index(drop=True)
)

print(f"Unique (price_date, ticker) combinations: {len(daily_returns):,}")
print(f"\nReturn statistics:")
print(daily_returns["next_day_ret"].describe())

# Compute return dispersion per day
def return_dispersion_metrics(group):
    """Compute cross-sectional return variance and other metrics for a day"""
    returns = group["next_day_ret"].dropna()
    if len(returns) >= MIN_TICKER_COVERAGE:
        return pd.Series({
            "return_var": returns.var(ddof=0),
            "return_std": returns.std(ddof=0),
            "return_mean": returns.mean(),
            "return_max": returns.max(),
            "return_min": returns.min(),
            "n_tickers_ret": len(returns),
        })
    else:
        return pd.Series({
            "return_var": np.nan,
            "return_std": np.nan,
            "return_mean": np.nan,
            "return_max": np.nan,
            "return_min": np.nan,
            "n_tickers_ret": len(returns),
        })

return_dispersion = daily_returns.groupby("price_date").apply(
    return_dispersion_metrics
).reset_index()
return_dispersion.columns = ["price_date"] + return_dispersion.columns[1:].tolist()

# Drop days with insufficient coverage
return_dispersion = return_dispersion[
    return_dispersion["n_tickers_ret"] >= MIN_TICKER_COVERAGE
].reset_index(drop=True)

print(f"\nReturn dispersion days (>=5 tickers): {len(return_dispersion):,}")
print(f"\nReturn dispersion statistics:")
print(return_dispersion[["return_var", "return_std", "return_mean"]].describe())

# Visualize return dispersion over time
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(return_dispersion["price_date"], return_dispersion["return_var"], color='darkred', linewidth=1, alpha=0.7)
axes[0].set_ylabel("Cross-sectional Variance", fontsize=11)
axes[0].set_title("Return Dispersion (Variance across MAG7)", fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(return_dispersion["price_date"], return_dispersion["return_std"], color='crimson', linewidth=1, alpha=0.7)
axes[1].set_ylabel("Cross-sectional Std Dev", fontsize=11)
axes[1].set_xlabel("Price Date", fontsize=11)
axes[1].set_title("Return Dispersion (Std Dev across MAG7)", fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Step 3 complete: Return dispersion metrics computed")

## Step 3: Compute Daily Return Dispersion Metrics

In [ ]:
# Step 1: Compute daily average sentiment per ticker
daily_sentiment = (
    df.groupby(["article_date", "ticker"])["sentiment_score"]
    .mean()
    .reset_index()
    .rename(columns={"sentiment_score": "mean_sentiment"})
)

print("\nDaily ticker-level sentiment aggregation:")
print(f"  Rows: {len(daily_sentiment):,}")
print(f"  Sample:")
print(daily_sentiment.head(10))

# Step 2: Compute sentiment dispersion metrics per day
def cross_sectional_variance(x):
    """Cross-sectional variance of sentiment across MAG7"""
    return x.var(ddof=0) if len(x) > 1 else np.nan

def sentiment_breadth(x):
    """% of MAG7 tickers with positive mean sentiment"""
    return (x > 0).mean() if len(x) > 0 else np.nan

def sentiment_mean(x):
    """Average sentiment across MAG7"""
    return x.mean() if len(x) > 0 else np.nan

sentiment_dispersion_all = daily_sentiment.groupby("article_date").agg(
    sentiment_var=("mean_sentiment", cross_sectional_variance),
    sentiment_breadth=("mean_sentiment", sentiment_breadth),
    sentiment_mean=("mean_sentiment", sentiment_mean),
    n_tickers=("ticker", "nunique"),
).reset_index()

print("\nSentiment dispersion (before coverage filter):")
print(f"  Days: {len(sentiment_dispersion_all):,}")
print(f"  Coverage distribution:")
print(f"    {sentiment_dispersion_all['n_tickers'].value_counts().sort_index().to_dict()}")

# Step 3: Apply minimum ticker coverage filter
sentiment_dispersion = sentiment_dispersion_all[
    sentiment_dispersion_all["n_tickers"] >= MIN_TICKER_COVERAGE
].copy().reset_index(drop=True)

print(f"\nAfter >=5 ticker coverage filter:")
print(f"  Days retained: {len(sentiment_dispersion):,} / {len(sentiment_dispersion_all):,}")
print(f"  Pct retained: {100*len(sentiment_dispersion)/len(sentiment_dispersion_all):.1f}%")
print(f"\nSentiment dispersion statistics:")
print(sentiment_dispersion[["sentiment_var", "sentiment_breadth", "sentiment_mean"]].describe())

# Visualize sentiment dispersion over time
fig, axes = plt.subplots(3, 1, figsize=(14, 8))

axes[0].plot(sentiment_dispersion["article_date"], sentiment_dispersion["sentiment_var"], color='steelblue', linewidth=1)
axes[0].set_ylabel("Cross-sectional Variance", fontsize=11)
axes[0].set_title("Sentiment Dispersion (Variance across MAG7)", fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(sentiment_dispersion["article_date"], sentiment_dispersion["sentiment_breadth"], color='darkorange', linewidth=1)
axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='50% breadth')
axes[1].set_ylabel("Sentiment Breadth", fontsize=11)
axes[1].set_title("Sentiment Breadth (% Positive across MAG7)", fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(sentiment_dispersion["article_date"], sentiment_dispersion["sentiment_mean"], color='green', linewidth=1)
axes[2].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[2].set_ylabel("Mean Sentiment", fontsize=11)
axes[2].set_xlabel("Article Date", fontsize=11)
axes[2].set_title("Average Sentiment across MAG7", fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Step 2 complete: Sentiment dispersion metrics computed")

## Step 2: Compute Daily Sentiment Dispersion Metrics

In [ ]:
# Load parquet file
df = pd.read_parquet("data/processed/gdelt_ohlcv_join.parquet")

print("=" * 70)
print("DATA LOADING")
print("=" * 70)
print(f"\n✓ Loaded {len(df):,} rows from gdelt_ohlcv_join.parquet")
print(f"Columns: {df.columns.tolist()}")
print(f"Date range (raw): {df['article_date'].min().date()} to {df['article_date'].max().date()}")
print(f"Tickers (raw): {sorted(df['ticker'].unique())}")

# Data filtering pipeline
print("\n" + "=" * 70)
print("DATA FILTERING PIPELINE")
print("=" * 70)

print(f"\n1. Starting rows: {len(df):,}")

# Step 1: Filter to MAG7
df = df[df["ticker"].isin(MAG7)].copy()
print(f"2. After MAG7 filter: {len(df):,} rows")

# Step 2: Filter to date window
df = df[
    (df["article_date"] >= WINDOW_START) &
    (df["article_date"] <= WINDOW_END)
].copy()
print(f"3. After date window ({WINDOW_START} to {WINDOW_END}): {len(df):,} rows")

# Step 3: Filter to sentiment_present only
df = df[df["sentiment_present"] == True].copy()
print(f"4. After sentiment_present filter: {len(df):,} rows")

print(f"\n   Unique article dates: {df['article_date'].nunique()}")
print(f"   Unique tickers: {sorted(df['ticker'].unique())}")
print(f"   Sentiment score range: [{df['sentiment_score'].min():.3f}, {df['sentiment_score'].max():.3f}]")
print(f"   Mean sentiment: {df['sentiment_score'].mean():.3f}, Std: {df['sentiment_score'].std():.3f}")

print("=" * 70)

## Step 1: Load and Filter Data

In [ ]:
import os
from pathlib import Path
from scipy.stats import spearmanr, mannwhitneyu
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set working directory to repo root
REPO_ROOT = Path.home() / "market-sentiment-analysis-8"
os.chdir(REPO_ROOT)
print(f"Working directory: {os.getcwd()}")

# Configuration
MAG7 = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA"]
WINDOW_START = "2024-02-23"
WINDOW_END = "2026-02-23"
MIN_TICKER_COVERAGE = 5

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)

# Structural Analysis: Hypothesis Testing Report

**Epic**: Sprint 4 — Structural Relationship Analysis  
**Issue**: #78 - Hypothesis Testing Report  
**Data Source**: `gdelt_ohlcv_join.parquet` (trevor branch)

## Objective
Determine whether statistically meaningful structural relationships exist between:
- **Sentiment dispersion** (cross-sectional variance of sentiment across MAG7)
- **Return dispersion** (cross-sectional variance of returns across MAG7)
- **Pairwise return correlation** (correlation breakdown across MAG7)

## Assumptions
- **Data window**: 2024-02-23 to 2026-02-23 (2 years)
- **Aggregation**: Daily
- **Filtering**: sentiment_present only, MIN_TICKERS >= 5
- **Sentiment version**: v1 (already in data)
- **Metrics**: locked per `docs/metrics.md`

## Key Hypotheses to Test
1. **H1**: ↑ sentiment dispersion → ↑ return dispersion (Spearman correlation)
2. **H2**: ↑ sentiment dispersion → ↓ pairwise correlation (Spearman correlation)
3. **H3**: Top 20% sentiment dispersion days have significantly different returns (Mann-Whitney U)